Data collection from YouTube

In [12]:
from youtube_comment_downloader import YoutubeCommentDownloader
import pandas as pd
import re
import time

downloader = YoutubeCommentDownloader()

# ---------------------------
# VIDEOS
# ---------------------------
video_urls = [
    "https://www.youtube.com/watch?v=-RZ86OB9hw4",
    "https://www.youtube.com/watch?v=ds9pEAB72kI",
    "https://www.youtube.com/watch?v=ZidGozDhOjg&t=480s",
    "https://www.youtube.com/watch?v=WWloIAQpMcQ&t=157s",
    "https://www.youtube.com/watch?v=iHEWj9POttY",
    "https://www.youtube.com/watch?v=e_aeUx4qhZQ", 
    "https://www.youtube.com/watch?v=_clCzEnTvAE",
    "https://www.youtube.com/watch?v=G-294NywwZ4",
    "https://www.youtube.com/watch?v=cqsE5GpyLdg",
    "https://www.youtube.com/watch?v=E3ZkZNXIdVo",
    "https://www.youtube.com/watch?v=IDPDEKtd2yM",
    "https://www.youtube.com/watch?v=lXZ5Bo5lafA",
    "https://www.youtube.com/watch?v=z-IR48Mb3W0"
]

# ---------------------------
# SETTINGS
# ---------------------------
TARGET_TOTAL = 9000
PER_VIDEO_LIMIT = 800   # 🔥 THIS WAS MISSING IN YOUR CODE

# ---------------------------
# CLEANING
# ---------------------------
def remove_emojis(text):
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002700-\U000027BF"
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub(r'', text)


def is_high_stress(text):
    text = text.lower()

    if len(text.split()) < 5:
        return False

    high_keywords = [
        "anxious", "anxiety", "panic", "overwhelmed",
        "burnout", "depressed", "stress", "stressed",
        "hopeless", "breakdown", "exhausted",
        "crying", "mental health", "shaking",
        "panic attack", "overthinking"
    ]

    extreme_keywords = [
        "suicide", "kill myself", "want to die",
        "end my life", "no reason to live"
    ]

    if any(k in text for k in extreme_keywords):
        return True

    return any(k in text for k in high_keywords)


# ---------------------------
# DATA STORAGE
# ---------------------------
data = []
seen = set()

# ---------------------------
# SCRAPING
# ---------------------------
for url in video_urls:
    print(f"\nScraping: {url}")
    
    per_video_count = 0

    try:
        comments = downloader.get_comments_from_url(url)

        for comment in comments:
            text = comment.get("text", "")
            clean_text = remove_emojis(text).strip()

            if not clean_text:
                continue

            if "http" in clean_text:
                continue

            if clean_text in seen:
                continue

            if not is_high_stress(clean_text):
                continue

            data.append([clean_text, "High", url])
            seen.add(clean_text)

            per_video_count += 1

            # limit per video
            if per_video_count >= PER_VIDEO_LIMIT:
                print(f"Reached per-video limit: {PER_VIDEO_LIMIT}")
                break

            # total limit
            if len(data) >= TARGET_TOTAL:
                break

        if len(data) >= TARGET_TOTAL:
            break

        time.sleep(1)

    except Exception as e:
        print(f"Error in {url}: {e}")

# ---------------------------
# SAVE
# ---------------------------
df = pd.DataFrame(data, columns=["text", "label", "source"])

print("\nFinal collected:", len(df))
df.to_csv("youtube_high_stress.csv", index=False)


Scraping: https://www.youtube.com/watch?v=-RZ86OB9hw4

Scraping: https://www.youtube.com/watch?v=ds9pEAB72kI

Scraping: https://www.youtube.com/watch?v=ZidGozDhOjg&t=480s
Reached per-video limit: 800

Scraping: https://www.youtube.com/watch?v=WWloIAQpMcQ&t=157s
Reached per-video limit: 800

Scraping: https://www.youtube.com/watch?v=iHEWj9POttY

Scraping: https://www.youtube.com/watch?v=e_aeUx4qhZQ

Scraping: https://www.youtube.com/watch?v=_clCzEnTvAE

Scraping: https://www.youtube.com/watch?v=G-294NywwZ4

Scraping: https://www.youtube.com/watch?v=cqsE5GpyLdg

Scraping: https://www.youtube.com/watch?v=E3ZkZNXIdVo

Scraping: https://www.youtube.com/watch?v=IDPDEKtd2yM
Reached per-video limit: 800

Scraping: https://www.youtube.com/watch?v=lXZ5Bo5lafA

Scraping: https://www.youtube.com/watch?v=z-IR48Mb3W0
Reached per-video limit: 800

Final collected: 5206
